# Native crypto market-data capture and replay

**Activity: market-data fetching and offline inspection.** This notebook is the
bounded MRs 22–27 learning checkpoint. It calls installed package behavior and
records real Bitfinex R0 input. Coinbase uses authenticated Exchange `level3`
when the local Exchange key is available; otherwise the exact access blocker is
shown and independent synthetic lifecycle checks still run.

We will connect, persist, decode, inspect gaps, and replay saved bytes. The
result is visible resting orders within source coverage. It does not establish
queue priority, hidden liquidity or executable fills. Live recording creates new
history; the Binance historical workflow remains in `research-foundations.ipynb`.
See [the collector guide](../docs/data/l3-collectors.md) for protocol references,
limits, credentials, CLI commands and failure handling.

Bitfinex R0 and Coinbase Exchange remain the L3 focus. This same workflow also includes **OKX BTC-USDT spot L2** live capture and a historical archive. USDT is distinct from USD; aggregate levels are never represented as individual orders. OKX historical input has no sequence IDs and does not automatically repair live gaps. See the [OKX contract](../docs/data/okx-market-data.md).


## 1. Installed environment and explicit source selection

Run all cells with a fresh kernel through `examples/run_notebook.py`. No cell
changes `sys.path`, installs dependencies, prints credentials or starts a service.
Package source and lock identities distinguish this execution from prior runs.

In [1]:
import hashlib
import json
import sys
from datetime import UTC, datetime, timedelta
from pathlib import Path

import alpha_research
import pandas as pd

from alpha_research.data.live.config import CaptureConfig, CaptureRequest
from alpha_research.data.live.journal import CaptureJournal, CaptureRecord
from alpha_research.data.live.replay import CaptureReplay
from alpha_research.data.live.transport import VenueTransport
from alpha_research.data.venues.coinbase.auth import CoinbaseAccessError, CoinbaseCredentials
from alpha_research.data.venues.coinbase.book import CoinbaseBook
from alpha_research.data.venues.okx.book import OKXBook
from alpha_research.data.venues.okx.history_config import OKXHistoryConfig
from alpha_research.data.venues.okx.history_store import OKXArchiveStore
from alpha_research.data.venues.okx.history_transport import OKXHistoryTransport
from alpha_research.pipelines.acquisition import acquire_history
from alpha_research.pipelines.live import InstalledIdentity, collect
from alpha_research.serialization import DomainCodec

root = Path.cwd()
assert (root / "configs/live").is_dir(), "Run from the repository root"
codec = DomainCodec()
implementation = InstalledIdentity.sha256()
bitfinex = CaptureConfig.read(root / "configs/live/bitfinex-btc-usd.json")
coinbase = CaptureConfig.read(root / "configs/live/coinbase-btc-usd.json")
okx = CaptureConfig.read(root / "configs/live/okx-btc-usdt.json")
okx_history = OKXHistoryConfig.read(root / "configs/history/okx-btc-usdt.json")
assert okx.instrument_id == okx_history.instrument_id
pd.DataFrame([{"python": sys.version.split()[0], "package": alpha_research.__file__,
               "implementation_sha256": implementation,
               "lock_sha256": hashlib.sha256((root / "uv.lock").read_bytes()).hexdigest()}])

,python,package,implementation_sha256,lock_sha256
0,3.14.7,/private/tmp/crypto-l3-verification/consumer/l...,e00f84a13948b3cf89ed8308a83e8ca2a5ded54fb83633...,e6cd3a169ff4b6850bcb5b09e38c44e1179b0dc1fb7c6a...


## 2. Native instruments and connection requirements

`probe` checks current public listing availability before collection. These are
native BTC/USD spot listings with separate canonical identities. Their public
availability does not prove that this account can subscribe to Exchange Level 3.
Bitfinex is public; Coinbase requires Exchange key/secret/passphrase. CDP JSON is
ignored. Configured limits below bound storage, message receipt and state.

OKX uses the public JSON `books` channel: 400 aggregate price levels per side and a nominal 100 ms cadence. Its current listing probe verifies BTC-USDT, SPOT/live status and native currencies. It does not establish historical listing coverage. The authenticated VIP channels are not selected.


In [2]:
discovery = []
for config in (bitfinex, coinbase, okx):
    payload = VenueTransport(config).discover()
    discovery.append({"venue": config.venue, "representation": "L2" if config.venue == "okx" else "L3", "native_symbol": config.symbol,
                      "instrument_id": str(config.instrument_id), "channel": config.channel,
                      "duration_s": config.duration_seconds, "message_limit": config.max_messages,
                      "journal_bytes": config.max_bytes, "depth": config.depth,
                      "discovery_sha256": hashlib.sha256(payload).hexdigest()})
pd.DataFrame(discovery)

,venue,representation,native_symbol,instrument_id,channel,duration_s,message_limit,journal_bytes,depth,discovery_sha256
0,bitfinex,L3,tBTCUSD,c87d983f-e7ec-4e64-8d54-9f2be67a64b8,R0,20,10000,32000000,25,ad5c2c16d99b75060bdfcb43aa70faa233931623d8b953...
1,coinbase,L3,BTC-USD,218142de-d549-43bb-85ba-a174fd4f798f,level3,20,10000,32000000,0,d2bfd4363396ca895bed3a49a2e3f265c7c043efbb50b0...
2,okx,L2,BTC-USDT,4476df8e-6342-4417-839e-e41b88f41446,books,20,10000,32000000,400,5868aba36f44605b113b2602986f4e2b2e08c55b6bb638...


## 3. Real bounded recording

The receiver timestamps each complete frame and sends it to a bounded queue.
The writer fsyncs each journal line before decoding it. Acknowledgements,
heartbeats, schema frames and epoch boundaries are retained alongside market
frames. A UUID directory prevents same-day overwrite. Gap/retry reasons remain
visible even when a later epoch recovers. Each call ends after its configured
limits; nothing runs unattended after this cell.

Both Bitfinex and OKX below use the same `pipelines/live.py` collection lifetime, journal and raw store. OKX acknowledges its subscription then sends an in-stream snapshot. Quiet sockets use text `ping`/`pong`; each captured interval is bounded independently.


In [3]:
bitfinex_result = await collect(bitfinex, root / "data/bitfinex",
                                implementation_identity=implementation)
assert bitfinex_result["replay_parity"]
assert bitfinex_result["synchronized"] and not bitfinex_result["gaps"], bitfinex_result
okx_result = await collect(okx, root / "data/okx/live", implementation_identity=implementation)
assert okx_result["replay_parity"]
assert okx_result["synchronized"] and not okx_result["gaps"], okx_result
assert okx_result["orders"] is None and okx_result["representation"] == "L2"
{"bitfinex": bitfinex_result, "okx": okx_result}


{'bitfinex': {'venue': 'bitfinex',
  'symbol': 'tBTCUSD',
  'instrument_id': 'c87d983f-e7ec-4e64-8d54-9f2be67a64b8',
  'channel': 'R0',
  'started_ns': 1789531801467557000,
  'finished_ns': 1789531821633034000,
  'messages': 382,
  'updates': 376,
  'snapshots': 1,
  'heartbeats': 2,
  'depth': 25,
  'orders': 50,
  'max_orders': 100000,
  'gaps': [],
  'synchronized': True,
  'ended': True,
  'source_sequence': 379,
  'last_source_time': None,
  'book_identity': '61b8ea219edc4f6bece997e5f32775d448294f98455d5ab9da54cdab65ff70f0',
  'trace_identity': '49dfbe292f56ecab931b1b8b387bff2a134a24ab6c186015081f62188c393cc7',
  'epochs': 1,
  'request_identity': '7e1062fd490d20edcc3eb0cfb59a9b5fabc83207393de526d37652ecc71c7d97',
  'saved_input_identity': 'bdad61d34dd3311d444701e6b71b31cedc991b8d5052629f2e71364603cebb0b',
  'saved_bytes': 53192,
  'capture_directory': '/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/data/bitfinex/613b3e58-f654-4420-87b3-601554977213',
  'sto

## 4. Coinbase access boundary

Fill `secrets/coinbase.secrets` locally using the three placeholders in the guide.
The loader rejects empty or malformed fields before opening a connection.
If eligible credentials are available this cell runs the same bounded collector;
otherwise it reports access-blocked status and continues to independent checks.
Credential values and signed outbound subscriptions are never displayed.

In [4]:
coinbase_result = None
try:
    exchange_credentials = CoinbaseCredentials.read(root / "secrets/coinbase.secrets")
except CoinbaseAccessError as error:
    coinbase_access = {"venue": "coinbase", "status": "access_blocked", "reason": str(error)}
else:
    coinbase_result = await collect(coinbase, root / "data/coinbase",
                                   implementation_identity=implementation,
                                   credentials=exchange_credentials)
    del exchange_credentials
    coinbase_access = {"venue": "coinbase", "status": "captured",
                       "synchronized": coinbase_result["synchronized"],
                       "gaps": coinbase_result["gaps"]}
coinbase_access

{'venue': 'coinbase',
 'status': 'access_blocked',
 'reason': 'Fill COINBASE_API_KEY, COINBASE_API_SECRET (base64), and COINBASE_API_PASSPHRASE in the Exchange secrets file'}

## 5. Meaningful interval and persisted-input identity

Receipt nanoseconds are UTC application observations, preserved without sorting.
Bitfinex supplies no source timestamp under the selected flags, so it remains
unknown. Depth 25 describes the requested raw-book window, not all venue orders.
Counts and saved identities below are derived from this run, not prefilled values.

OKX reports `levels`, `orders=None` and a distinct native symbol. Its aggregate order counts do not reveal individual identities or priority.


In [5]:
results = [bitfinex_result, okx_result] + ([] if coinbase_result is None else [coinbase_result])
rows = []
for result in results:
    row = {name: result[name] for name in ("venue", "symbol", "messages", "snapshots", "updates",
                                          "orders", "depth", "gaps", "saved_input_identity")}
    row["levels"] = result.get("levels")
    row["representation"] = "L2" if result["venue"] == "okx" else "L3"
    for name in ("started_ns", "finished_ns"):
        row[name.replace("_ns", "_utc")] = (
            datetime(1970, 1, 1, tzinfo=UTC) + timedelta(microseconds=result[name] // 1000)
        ).isoformat()
    rows.append(row)
pd.DataFrame(rows)

,venue,symbol,messages,snapshots,updates,orders,depth,gaps,saved_input_identity,levels,representation,started_utc,finished_utc
0,bitfinex,tBTCUSD,382,1,376,50.0,25,[],bdad61d34dd3311d444701e6b71b31cedc991b8d505262...,NaN,L3,2026-09-16T04:10:01.467557+00:00,2026-09-16T04:10:21.633034+00:00
1,okx,BTC-USDT,186,1,184,NaN,400,[],c3df4896f45eadaab1810dbbe8c9254bcb9e0feb1c487d...,800.0,L2,2026-09-16T04:10:21.644895+00:00,2026-09-16T04:10:41.732980+00:00


## 6. Inspect persistence and exact decoding

The manifest uses the existing raw-store SHA-256 identity. Reads verify all
published bytes before decoding. We display the receipt envelope and numeric
**types**, keeping bulk raw feed data under the ignored venue directory. Source
order IDs and decimal tokens are preserved in saved bytes. The replay consumer
uses the same venue decoder as collection.

For OKX, exact absolute sizes replace prior levels and zero size deletes a price. The bounded state keeps the best 400 prices per side. `prevSeqId` must match the previous `seqId`; numeric jumps, empty repeated sequences and linked maintenance resets follow the venue protocol. Deprecated checksums remain raw metadata. Source timestamps remain supplied Unix-millisecond text or unknown.


In [6]:
capture_root = Path(bitfinex_result["capture_directory"])
request = codec.decode_json((capture_root / "request.json").read_bytes(), CaptureRequest)
journal = CaptureJournal(capture_root, request)
first_frame = next(record for record in journal.records() if record.kind == "frame")
offline = CaptureReplay.read(capture_root)
example_order = next(iter(offline.book.orders.values()))
okx_root = Path(okx_result["capture_directory"])
okx_offline = CaptureReplay.read(okx_root)
assert isinstance(okx_offline.book, OKXBook)
okx_price, okx_level = next(iter(okx_offline.book.bids.items()))
pd.DataFrame([
    {"venue": "bitfinex", "epoch": first_frame.epoch, "receipt_ns": first_frame.received_ns,
     "raw_frame_bytes": len(first_frame.raw),
     "raw_frame_sha256": hashlib.sha256(first_frame.raw).hexdigest(),
     "order_id_type": type(example_order.order_id).__name__,
     "price_type": type(example_order.price).__name__, "size_type": type(example_order.size).__name__,
     "source_time": offline.book.source_time},
    {"venue": "okx", "order_id_type": None, "price_type": type(okx_price).__name__,
     "size_type": type(okx_level.size).__name__, "reported_orders_at_price": okx_level.order_count,
     "source_time": okx_offline.book.source_time, "source_time_unit": "unix_milliseconds"},
])


,venue,epoch,receipt_ns,raw_frame_bytes,raw_frame_sha256,order_id_type,price_type,size_type,source_time,reported_orders_at_price,source_time_unit
0,bitfinex,0.0,1.789532e+18,102.0,06e9379502b7c6f703950eeaeed80bf278a966581d0882...,str,Decimal,Decimal,NaN,NaN,NaN
1,okx,NaN,NaN,NaN,NaN,NaN,Decimal,Decimal,1789531841501,1.0,unix_milliseconds


## 7. Offline replay and a synthetic gap exercise

No network is needed to reopen `capture_root`. The rolling trace compares state
at every source-message boundary, including controls and receipt clocks. The
final state and trace must equal collection. A checksum identity alone does not
make missing input retrievable: keep the local directory together with its
manifest and request file.

**Exercise:** append a synthetic disconnect to the observed prefix. Does a later
heartbeat alone make the book usable? The answer is no: the affected epoch stays
invalid until a fresh subscription and snapshot. This exercise does not alter
the real saved input.

In [7]:
assert offline.summary()["trace_identity"] == bitfinex_result["trace_identity"]
assert offline.summary()["book_identity"] == bitfinex_result["book_identity"]
assert offline.messages == bitfinex_result["messages"]
assert okx_offline.summary()["trace_identity"] == okx_result["trace_identity"]
assert okx_offline.summary()["book_identity"] == okx_result["book_identity"]
exercise = CaptureReplay(bitfinex)
for record in journal.records():
    if record.kind == "end":
        break
    exercise.apply(record)
exercise.apply(CaptureRecord(kind="gap", epoch=exercise.epoch,
                             received_ns=exercise.finished_ns, reason="synthetic_disconnect"))
assert not exercise.book.ready
{"offline_parity": True, "okx_offline_parity": True, "verified_messages": offline.messages,
 "trace_sha256": offline.summary()["trace_identity"],
 "synthetic_gap": exercise.gaps[-1], "usable_after_gap": exercise.book.ready}

{'offline_parity': True,
 'okx_offline_parity': True,
 'verified_messages': 382,
 'trace_sha256': '49dfbe292f56ecab931b1b8b387bff2a134a24ab6c186015081f62188c393cc7',
 'synthetic_gap': {'epoch': 0,
  'received_ns': 1789531821632238000,
  'reason': 'synthetic_disconnect'},
 'usable_after_gap': False}

## 8. Coinbase snapshot bridge and lifecycle, independent of account access

This final case is explicitly synthetic. It uses the official compact schema
shape and a hand-authored full L3 snapshot at sequence 10. A queued `noop` at 11
bridges the snapshot; then a maker match at 12 reduces exact resting size.
Received/non-resting events must not be treated as visible opens. Synthetic
checks establish software behavior; they do not establish authenticated source
access, queue priority or hypothetical execution quality.

In [8]:
coinbase_book = CoinbaseBook(coinbase)
coinbase_book.apply((root / "tests/fixtures/l3/coinbase-schema.json").read_bytes())
coinbase_book.apply(b'{"type":"subscriptions","channels":[{"name":"level3","product_ids":["BTC-USD"]}]}')
coinbase_book.apply(b'["noop","BTC-USD","11",null]')
coinbase_book.apply((root / "tests/fixtures/l3/coinbase-snapshot.json").read_bytes(), snapshot=True)
coinbase_book.apply(b'["match","BTC-USD","12","maker","taker","100","0.1",null]')
assert str(coinbase_book.orders["maker"].size) == "0.200000000000000001"
assert coinbase_book.ready
{"evidence": "synthetic", "sequence": coinbase_book.sequence,
 "remaining_size": str(coinbase_book.orders["maker"].size),
 "orders": len(coinbase_book.orders), "source_time": coinbase_book.source_time,
 "authenticated_live_evidence": coinbase_access["status"]}

{'evidence': 'synthetic',
 'sequence': 12,
 'remaining_size': '0.200000000000000001',
 'orders': 1,
 'source_time': None,
 'authenticated_live_evidence': 'access_blocked'}

## 9. Discover and download one historical UTC day

The maintained example selects **8 September 2026** with module `4` (400-level L2), daily aggregation and exactly one native instrument. The public history API uses inclusive UTC dates for order books; release is typically T+3 and availability varies. The index and precise CDN URL are retained with their receipt time and hash.

The existing `pipelines/acquisition.py` owns this historical operation. Use `alpha-acquire archive index/fetch/replay` from the installed command. This cell downloads the complete archive (observed size about 104 MiB), verifies gzip CRC, expanded size and the single native tar member, then publishes its original bytes with the shared raw store. It never extracts archive paths. Configured caps are 160 MB compressed and 1 GB expanded; expansion is streamed. A partial download has no completion manifest. Retry starts the download again within finite limits.


In [9]:
index = OKXHistoryTransport(okx_history).index()
url = okx_history.select_url(index)
print({"utc_day": okx_history.day.isoformat(), "module": 4, "url": url,
       "compressed_limit": okx_history.max_bytes,
       "expanded_limit": okx_history.max_expanded_bytes,
       "replay_record_limit": okx_history.max_records})
history_result = acquire_history(okx_history, root / "data/okx/history",
                                 implementation_identity=implementation)
assert history_result["state_available"]
history_result


{'utc_day': '2026-09-08', 'module': 4, 'url': 'https://static.okx.com/cdn/okx/match/orderbook/pro/L2/400lv/daily/20260908/BTC-USDT-L2orderbook-400lv-2026-09-08.tar.gz', 'compressed_limit': 160000000, 'expanded_limit': 1000000000, 'replay_record_limit': 10000}


{'venue': 'okx',
 'symbol': 'BTC-USDT',
 'instrument_id': '4476df8e-6342-4417-839e-e41b88f41446',
 'representation': 'L2',
 'mode': 'historical',
 'depth': 400,
 'orders': None,
 'records': 10000,
 'snapshots': 1,
 'updates': 9999,
 'levels': 800,
 'source_sequence': None,
 'first_source_ms': '1788825600004',
 'last_source_ms': '1788825824624',
 'record_receipt_time': None,
 'continuity': 'unproven_no_source_sequences',
 'gaps': None,
 'state_available': True,
 'expanded_bytes': 764149760,
 'stop_reason': 'record_limit',
 'entire_archive_replayed': False,
 'book_identity': '98463d3904ce63326fb0e2a9bab95b927a6068289aa1989cebe66e18fabdf315',
 'trace_identity': 'e8c58ef9bc585227184ea43f5ced9ded45ba451ea91b0251544056ef5c5e6811',
 'saved_input_identity': 'fef59b343d1972c32f67583edc538c5d2f7350e6f3a0059e6f87896d7e8bf676',
 'saved_bytes': 109423187,
 'request_identity': '503f9033527bc16073e080159a647ed06814e89345d4e0e71db755b780744469',
 'index_identity': 'c02a0194941bfd72bf14d5609381c8fe791a

## 10. Replay an explicit historical prefix offline

The observed archive contains three-field levels (`price`, `size`, `order_count`) in newline-delimited snapshots/updates. Unlike live frames, these records have **no sequence IDs and no original receipt timestamps**. They are not synthesized. Reported `gaps=None` means continuity cannot be established from source sequencing; it does not mean zero gaps.

The complete compressed day is retained, but this notebook reconstructs only the first **10,000 records**. Every replay verifies the entire raw digest and archive integrity before applying that same prefix. The source interval below describes the processed prefix, not a full day. Increasing the maintained record limit selects more rows on a subsequent download; this bounded example does not claim full-day semantic validation or normalized dataset publication.


In [10]:
offline_history = OKXArchiveStore(Path(history_result["archive_directory"])).replay()
assert offline_history["trace_identity"] == history_result["trace_identity"]
assert offline_history["book_identity"] == history_result["book_identity"]
assert offline_history["records"] == history_result["records"]
assert offline_history["source_sequence"] is None and offline_history["gaps"] is None
pd.DataFrame([{key: offline_history[key] for key in (
    "venue", "symbol", "records", "snapshots", "updates", "levels", "depth",
    "first_source_ms", "last_source_ms", "continuity", "entire_archive_replayed",
    "stop_reason", "saved_input_identity", "trace_identity")}])


,venue,symbol,records,snapshots,updates,levels,depth,first_source_ms,last_source_ms,continuity,entire_archive_replayed,stop_reason,saved_input_identity,trace_identity
0,okx,BTC-USDT,10000,1,9999,800,400,1788825600004,1788825824624,unproven_no_source_sequences,False,record_limit,fef59b343d1972c32f67583edc538c5d2f7350e6f3a005...,e8c58ef9bc585227184ea43f5ced9ded45ba451ea91b02...


## 11. One evidence table across the configured sources

Receipt intervals describe live recordings; source intervals describe the explicit historical prefix. Coinbase access status remains reported in section 4.


In [11]:
epoch = datetime(1970, 1, 1, tzinfo=UTC)
evidence = []
for result in results:
    evidence.append({"venue": result["venue"], "symbol": result["symbol"], "mode": "live",
        "representation": "L2" if result["venue"] == "okx" else "L3",
        "start_utc": (epoch + timedelta(microseconds=result["started_ns"] // 1000)).isoformat(),
        "end_utc": (epoch + timedelta(microseconds=result["finished_ns"] // 1000)).isoformat(),
        "clock": "receipt", "records": result["messages"], "orders": result["orders"],
        "levels": result.get("levels"), "depth": result["depth"], "gaps": result["gaps"],
        "saved_input_sha256": result["saved_input_identity"]})
evidence.append({"venue": "okx", "symbol": okx_history.symbol, "mode": "historical prefix",
    "representation": "L2", "clock": "source", "records": offline_history["records"],
    "start_utc": (epoch + timedelta(milliseconds=int(offline_history["first_source_ms"]))).isoformat(),
    "end_utc": (epoch + timedelta(milliseconds=int(offline_history["last_source_ms"]))).isoformat(),
    "orders": None, "levels": offline_history["levels"], "depth": 400, "gaps": None,
    "saved_input_sha256": offline_history["saved_input_identity"]})
pd.DataFrame(evidence)


,venue,symbol,mode,representation,start_utc,end_utc,clock,records,orders,levels,depth,gaps,saved_input_sha256
0,bitfinex,tBTCUSD,live,L3,2026-09-16T04:10:01.467557+00:00,2026-09-16T04:10:21.633034+00:00,receipt,382,50.0,NaN,25,[],bdad61d34dd3311d444701e6b71b31cedc991b8d505262...
1,okx,BTC-USDT,live,L2,2026-09-16T04:10:21.644895+00:00,2026-09-16T04:10:41.732980+00:00,receipt,186,NaN,800.0,400,[],c3df4896f45eadaab1810dbbe8c9254bcb9e0feb1c487d...
2,okx,BTC-USDT,historical prefix,L2,2026-09-08T00:00:00.004000+00:00,2026-09-08T00:03:44.624000+00:00,source,10000,NaN,800.0,400,None,fef59b343d1972c32f67583edc538c5d2f7350e6f3a005...


## Interpretation and stopping point

The Bitfinex success path above is a real bounded capture with verified offline
parity. Coinbase live status is stated separately. Malformed schemas, numeric
errors, missing auth, bridge/sequence gaps, disconnect/recovery, queue overflow,
byte limits and interrupted persistence also have maintained deterministic tests.
The current reconstruction supports visible order observations only. Canonical
dataset publication, generalized replay scheduling, external supervision and all
execution modeling remain with their roadmap owners. Close the kernel when done.

OKX historical integrity and prefix reconstruction are verified separately from continuity, which remains unproven without source sequences. The public `books` channel excludes ELP/RPI liquidity; no equivalence of archive and live sampling is assumed. No queue priority or fill model is inferred. All sources remain bounded foreground work.
